# Extracellular matrix gene expression across brain regions

Extracts and reports extracellular matrix (ECM) gene expression per Desikan-Killiany region from the Allen
Human Brain Atlas, reusing the same infrastructure as `alzheimers_selective_vulnerability.ipynb` and
`scaled_selective_vulnerability.ipynb` in this directory.

**Run this in Google Colab.** Not run here: this sandbox's network policy blocks `api.brain-map.org`
(AHBA), confirmed by direct test in this session (again, just before writing this notebook — same result
as every earlier check this session). It also blocks every standard ECM/gene-ontology annotation source I
tried just now: `matrisomeproject.mit.edu`, `mygene.info`, `geneontology.org`, `www.gsea-msigdb.org`,
`maayanlab.cloud` — all `403 connect_rejected`, same policy pattern as the AHBA/GWAS/neuromaps-surface
hosts in the earlier notebooks.

**On the gene list, specifically:** `abagen` ships a curated-gene-group fetcher
(`abagen.fetch_gene_group`), confirmed in this session to support exactly five groups — `brain`, `neuron`,
`oligodendrocyte`, `synaptome`, `layers` — and no ECM/matrisome option. I'm not substituting a hand-typed
ECM gene list from memory for it: an unverified guess at "the" extracellular matrix gene set is a much
bigger claim than the single unverified EFO code flagged in the first notebook, and this notebook follows
the same rule — fetch it from a citable source at runtime, in an environment where that source is actually
reachable, rather than assert it here. Section 1 below does that, against the field-standard curated
resource for this specific question.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/ahba_alzheimers'  # same directory as the other two notebooks
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
!pip install -q abagen nilearn openpyxl mygene

## 1. Get the ECM gene list

**Primary source: the Matrisome Project** (Naba et al.; Hynes Lab, MIT — matrisomeproject.mit.edu), the
field-standard curated human "matrisome" gene list, split into *Core matrisome* (collagens, ECM
glycoproteins, proteoglycans) and *Matrisome-associated* (ECM regulators, ECM-affiliated proteins, secreted
factors). Go to matrisomeproject.mit.edu, find the current human masterlist download (Excel or CSV) on
their Downloads page, and place it at `MATRISOME_PATH` below.

I'm deliberately not hardcoding the exact current filename/URL — `matrisomeproject.mit.edu` was unreachable
from this session to confirm it, and this project has revised the masterlist across versions before, so a
guessed link risks being silently stale. Same reasoning as the GWAS Catalog bulk-download step in the
scaled-up notebook.

The column-detection code below inspects the file's actual columns instead of assuming exact names, for the
same reason — I can't check the current schema live. **Read the printed columns before trusting the
auto-detected ones.**

In [ ]:
import pandas as pd

MATRISOME_PATH = f'{DATA_DIR}/Hs_Matrisome_Masterlist.xlsx'  # adjust extension/name to what you downloaded

matrisome = pd.read_excel(MATRISOME_PATH) if MATRISOME_PATH.endswith(('.xlsx', '.xls')) else pd.read_csv(MATRISOME_PATH)
print("Columns found:", matrisome.columns.tolist())
matrisome.head()

In [ ]:
# Auto-detect the gene-symbol column; STOP and set gene_col manually if this looks wrong.
candidates = [c for c in matrisome.columns if 'gene' in c.lower() and 'symbol' in c.lower()]
if not candidates:
    candidates = [c for c in matrisome.columns if c.lower().strip() == 'symbol']
assert candidates, f"Couldn't auto-detect a gene-symbol column among {matrisome.columns.tolist()} — set gene_col manually."
gene_col = candidates[0]
print(f"Using gene-symbol column: {gene_col!r}")

# Auto-detect the Core/Associated division column the same way.
division_candidates = [c for c in matrisome.columns if 'division' in c.lower()]
division_col = division_candidates[0] if division_candidates else None
if division_col:
    print(f"Using division column: {division_col!r}, values: {matrisome[division_col].dropna().unique().tolist()}")
else:
    print("No division column auto-detected — every matched gene will be treated as one undivided ECM set.")

In [ ]:
INCLUDE_DIVISIONS = None  # e.g. ['Core matrisome', 'Matrisome-associated'] once you've seen the real values above; None = include every row

ecm_table = matrisome if (division_col is None or INCLUDE_DIVISIONS is None) else matrisome[matrisome[division_col].isin(INCLUDE_DIVISIONS)]
ecm_genes = sorted(ecm_table[gene_col].dropna().astype(str).str.strip().unique().tolist())
print(f"{len(ecm_genes)} ECM gene symbols loaded from the Matrisome masterlist.")

### 1b. Independent cross-check — GO:0031012 "extracellular matrix" via MyGene.info

A second, independently-curated source, queried live rather than typed from memory. **The exact query
syntax below (`go.CC.id:GO:0031012`) reflects MyGene.info's documented field-query convention as I
understand it, but wasn't verifiable live from this session** (`mygene.info` was blocked here too) —
confirm it against https://mygene.info/doc/query_service.html if it returns zero or clearly-wrong results
in Colab, rather than assuming the fault is in your data. This is a cross-check, not the primary list: the
overlap between the two tells you whether the Matrisome list and a plain GO-term annotation agree, which is
worth knowing before treating either as ground truth on its own.

In [ ]:
import mygene

mg = mygene.MyGeneInfo()
go_hits = mg.query('go.CC.id:GO:0031012', species='human', fields='symbol', size=1000, fetch_all=True)
go_ecm_genes = sorted({hit['symbol'] for hit in go_hits if 'symbol' in hit})
print(f"{len(go_ecm_genes)} genes from the GO:0031012 cross-check.")

overlap = set(ecm_genes) & set(go_ecm_genes)
print(f"Overlap with the Matrisome list: {len(overlap)} genes "
      f"({len(overlap) / max(len(ecm_genes), 1):.0%} of the Matrisome list, "
      f"{len(overlap) / max(len(go_ecm_genes), 1):.0%} of the GO list)")
print("Low overlap is a real, informative result here (the two resources define 'ECM' differently, "
      "Matrisome deliberately more broadly) — not necessarily a bug in either list.")

## 2. Get the AHBA expression data (reuse)

Same as the other two notebooks: loads the cached `ahba.csv` if you've already run either of them against
this `DATA_DIR`, otherwise downloads it (~4GB, Colab only).

In [ ]:
import abagen

atlas = abagen.fetch_desikan_killiany()
info = pd.read_csv(atlas['info'])

HEMISPHERE_MODE = "left_only"  # keep consistent with the other two notebooks

ahba_cache = f'{DATA_DIR}/ahba.csv'
if os.path.exists(ahba_cache):
    expression = pd.read_csv(ahba_cache, index_col=0)
    expression.columns = expression.columns.astype(str)
    print(f"Loaded cached expression data: {expression.shape}")
else:
    if HEMISPHERE_MODE == "left_only":
        expression = abagen.get_expression_data(atlas['image'], atlas['info'], lr_mirror=None, data_dir=DATA_DIR)
        keep_ids = info.loc[info['hemisphere'].isin(['L', 'B']), 'id']
        expression = expression.loc[expression.index.isin(keep_ids)]
    else:
        expression = abagen.get_expression_data(atlas['image'], atlas['info'], lr_mirror='bidirectional', data_dir=DATA_DIR)
    expression.to_csv(ahba_cache)
    print(f"Downloaded and cached expression data: {expression.shape}")

## 3. Match ECM genes to the expression data

In [ ]:
available_ecm = [g for g in ecm_genes if g in expression.columns]
print(f"{len(available_ecm)} of {len(ecm_genes)} Matrisome ECM genes found in the AHBA expression data")
if len(available_ecm) < 0.5 * len(ecm_genes):
    print("WARNING: fewer than half matched — check gene symbol formatting/aliases before trusting the extraction.")

## 4. Extract: full gene x region matrix, and a per-region summary score

This is the actual extraction: every matched ECM gene's expression in every region, saved as its own file
— not just a single collapsed score — plus a per-region mean as a simple summary for a first look and for
the plot in Section 5.

In [ ]:
ecm_expression = expression[available_ecm].copy()
ecm_expression = ecm_expression.merge(info[['id', 'label', 'hemisphere', 'structure']],
                                        left_index=True, right_on='id').set_index('label')
ecm_expression.to_csv(f'{DATA_DIR}/ecm_expression_by_region.csv')
print(f"Saved full gene x region ECM expression matrix: {ecm_expression.shape} -> {DATA_DIR}/ecm_expression_by_region.csv")

region_summary = pd.DataFrame({
    'mean_ecm_expression': expression[available_ecm].mean(axis=1),
    'median_ecm_expression': expression[available_ecm].median(axis=1),
})
region_summary = region_summary.merge(info[['id', 'label', 'hemisphere', 'structure']],
                                        left_index=True, right_on='id').set_index('label')
region_summary = region_summary.sort_values('mean_ecm_expression', ascending=False)
region_summary.to_csv(f'{DATA_DIR}/ecm_region_summary.csv')
region_summary

If a `division_col` was detected in Section 1, this also breaks the summary out by Core vs.
Matrisome-associated — collagens/glycoproteins/proteoglycans behave differently across the brain than ECM
regulators and secreted factors do, and collapsing them into one number hides that.

In [ ]:
if division_col is not None and INCLUDE_DIVISIONS is None:
    by_division = {}
    for div_value in matrisome[division_col].dropna().unique():
        div_genes = sorted(matrisome.loc[matrisome[division_col] == div_value, gene_col].dropna().astype(str).str.strip().unique())
        div_available = [g for g in div_genes if g in expression.columns]
        if div_available:
            by_division[div_value] = expression[div_available].mean(axis=1)
    division_summary = pd.DataFrame(by_division)
    division_summary = division_summary.merge(info[['id', 'label']], left_index=True, right_on='id').set_index('label')
    division_summary.to_csv(f'{DATA_DIR}/ecm_region_summary_by_division.csv')
    division_summary
else:
    print("Skipped: no division column detected, or INCLUDE_DIVISIONS was already narrowed in Section 1.")

## 5. Plot

In [ ]:
from nilearn import plotting

plotting.plot_roi(
    atlas['image'],
    title="Mean extracellular matrix gene expression by region",
)

## 6. Optional: is ECM expression regionally enriched anywhere, beyond chance?

Section 4 reports raw levels — this asks the sharper question, reusing the same permutation-test machinery
as Part 3 of the single-disease notebook: is the ECM gene set's expression in any region higher than
10,000 random gene sets of the same size would produce there? Unlike the disease notebooks, there's no
single anatomical target to check this against (ECM genes aren't associated with one damage site the way
Alzheimer's risk genes are) — this reports the full ranked table instead of checking a specific region.

In [ ]:
import numpy as np
from statsmodels.stats.multitest import multipletests

rng = np.random.default_rng(0)
all_genes = expression.columns.to_numpy()
n_perm = 10000
observed = expression[available_ecm].mean(axis=1)
null = np.zeros((n_perm, len(expression)))
for i in range(n_perm):
    fake = rng.choice(all_genes, size=len(available_ecm), replace=False)
    null[i] = expression[fake].mean(axis=1)

z = (observed - null.mean(axis=0)) / null.std(axis=0)
p = (null >= observed.to_numpy()).mean(axis=0)
_, p_fdr, _, _ = multipletests(p, method='fdr_bh')

enrichment = pd.DataFrame({'region': expression.index, 'z': z, 'p_fdr': p_fdr})
enrichment = enrichment.merge(info[['id', 'label', 'hemisphere', 'structure']], left_on='region', right_on='id')
enrichment.sort_values('z', ascending=False)

## What's left to write up

1. Which Matrisome masterlist version you downloaded (Section 1) and its exact column names — the
   auto-detection is a convenience, not a substitute for checking it matched the right columns.
2. The overlap between the Matrisome list and the GO:0031012 cross-check (Section 1b) — low overlap is a
   real result about how differently the two resources scope "ECM," not an error to explain away.
3. How many ECM genes matched the AHBA data (Section 3) — same caveat as both other notebooks: a low match
   fraction means check gene symbol aliasing before trusting anything downstream.
4. Whether any region's ECM enrichment (Section 6) survives FDR correction — and per the pattern established
   in the other two notebooks, this hasn't been checked against a spatial-autocorrelation-corrected null
   (`neuromaps`); treat the naive z/p-values here with the same caution Part 5 of the single-disease notebook
   argues for.
5. The six-donor AHBA sample size, same limitation as both other notebooks, not something this one fixes
   either.